# VAST Challenge 2026 — Mini-Challenge 2
## Análise visual e narrativa da cadeia anômala de publicação no SaidIt

**Autores:** Esdras Silva (202411140033) e João Pedro Silva (202411140020)

Este notebook documenta a metodologia, o pré-processamento e as evidências que sustentam as respostas ao Mini-Challenge 2. A análise foi organizada como uma investigação orientada por perguntas, de modo que cada etapa produza uma evidência visual e conduza à etapa seguinte.

### Pergunta central

> Como um conteúdo armazenado em `SwiftWren.txt` chegou ao SaidIt em 17 de maio de 2046, por que os arquivos relacionados foram removidos logo depois e onde seria possível intervir com o menor impacto?

### Estrutura narrativa da investigação

1. **Incidente:** identificar o evento principal e seus elementos básicos;
2. **Raridade:** verificar se o comportamento aparece no baseline do ambiente;
3. **Sistema:** contextualizar agentes, arquivos e sistemas envolvidos;
4. **Cadeia:** reconstruir a sequência temporal do SwiftWren;
5. **Origem:** distinguir origem operacional, processual e significado provável;
6. **Comparação:** procurar casos anteriores e caracterizar padrões, tendências e outliers;
7. **Intervenção:** selecionar um único ponto de controle;
8. **Respostas:** consolidar as conclusões do desafio.

A narrativa não substitui a análise exploratória. Ela organiza os resultados em uma sequência causal e argumentativa, enquanto o dashboard Streamlit preserva filtros e inspeção livre dos registros.


## Metodologia analítica e estratégia de storytelling

A metodologia combina análise de dados, visualização interativa e comunicação narrativa.

### Fluxo metodológico

**Aquisição e validação → normalização → baseline → detecção de casos → reconstrução temporal → proveniência → comparação → intervenção**

- **Aquisição e validação:** leitura do ZIP original e verificação das colunas essenciais;
- **Normalização:** extração de atributos do campo `details`, padronização de agentes, arquivos e timestamps;
- **Baseline:** comparação entre postagens comuns e postagens com `content_source`;
- **Detecção:** identificação dos casos HiddenOrca, MellowOtter e SwiftWren;
- **Reconstrução:** ordenação dos eventos e análise dos segundos finais;
- **Proveniência:** associação entre arquivo de instruções, arquivo de conteúdo, cadeia de agentes e publicação;
- **Comparação:** medidas de duração, delegações, agentes e exclusões;
- **Intervenção:** avaliação da convergência dos casos para `system:saidit`.

### Princípio de comunicação

Cada seção do notebook segue quatro elementos:

1. **Pergunta investigativa** — o que precisa ser descoberto;
2. **Evidência** — quais registros ou métricas são observados;
3. **Interpretação** — o que os dados sustentam;
4. **Limitação** — o que não pode ser afirmado com segurança.

Essa estrutura reduz afirmações não sustentadas e mantém separadas evidências diretas e inferências comportamentais.


## 1. Configuração e carregamento dos dados

### Pergunta investigativa
Como preparar os registros sem perder a informação temporal original?

O conjunto é lido diretamente do ZIP distribuído para o MC2. Os timestamps UTC são preservados e uma coluna UTC−7 é criada apenas para reproduzir a escala temporal adotada na investigação.

### Evidência e decisão metodológica
A análise utiliza os tempos originais para cálculos e a representação UTC−7 para comunicação. O deslocamento não é tratado como inferência geográfica.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_local_dataset
from src.preprocessing import preprocess_events
from src.analysis import *

raw, org = load_local_dataset(PROJECT_ROOT / "data" / "VAST_Challenge_2026_MC2.zip")
df = preprocess_events(raw)
posts = file_based_posts(df)
summary = comparison_table(df, posts)
baseline = post_baseline(df)
print(df.shape)
posts[["case_name", "datetime_local_desafio", "forum", "content_source", "source_entity"]]

## 2. O comportamento é raro?

### Pergunta investigativa
O uso de arquivo como fonte de conteúdo e a exclusão rápida dos artefatos são comportamentos comuns no SaidIt?

O baseline compara todas as publicações do sistema com as postagens que usam `content_source` e apresentam exclusões em até cinco segundos.

### Evidência esperada
Se a combinação ocorrer em poucos casos, ela funciona como sinal discriminante para orientar a investigação.


In [ ]:
baseline.groupby(["usa_content_source", "delete_file_ate_5s"]).size().reset_index(name="quantidade")

In [ ]:
anomaly_funnel(df, posts)

## 3. Reconstrução do caso principal: SwiftWren

### Pergunta investigativa
Qual sequência de eventos antecedeu e sucedeu a publicação de `SwiftWren.txt`?

O evento principal ocorreu em 17/05/2046 às 04:21:15 no horário analítico UTC−7. Os registros são ordenados para reconstruir a cadeia completa e, em seguida, ampliar os segundos finais.

### Interpretação
A cadeia longa contextualiza a propagação, mas os eventos imediatamente próximos ao post fornecem a evidência mais direta do desfecho.


In [ ]:
case_name = "SwiftWren"
events = case_events(df, case_name)
post = post_for_case(posts, case_name)
seq = final_sequence(events, n_delegations=20)
seq[["ordem", "datetime_local_desafio", "short_name", "descricao", "delta_s", "path", "content_source"]]

## 4. De onde veio o conteúdo?

### Pergunta investigativa
Quais arquivos e processos deram origem à publicação e até que ponto é possível inferir seu significado?

A origem operacional do post é `SwiftWren.txt`. A origem processual está associada a `SwiftWren_further_instructions.md` e à cadeia de delegações.

### Limitação
O conteúdo literal não está preservado nos logs. Portanto, a análise descreve o comportamento de produção, propagação, publicação e remoção, sem afirmar as palavras exatas da mensagem.


In [ ]:
provenance_table(df, "SwiftWren")

## 5. SwiftWren foi um caso isolado?

### Pergunta investigativa
Existem ocorrências anteriores com a mesma combinação de sinais?

`HiddenOrca`, `MellowOtter` e `SwiftWren` compartilham o mesmo mecanismo final, embora tenham escalas diferentes.

### Elementos comparados
- quantidade de eventos;
- número de delegações;
- duração da cadeia;
- agentes distintos;
- uso de `content_source`;
- exclusões posteriores;
- agente responsável pela publicação.

A comparação permite identificar padrões recorrentes e destacar SwiftWren como outlier quantitativo.


In [ ]:
summary

In [ ]:
similarity_table(df, posts)

## 6. Onde intervir?

### Pergunta investigativa
Qual ponto único pode cobrir os três casos sem bloquear toda a comunicação entre agentes?

O ponto recomendado é `system:saidit`, imediatamente antes da publicação. A regra deve considerar a combinação de sinais, e não apenas a presença de um arquivo `.txt`.

### Critérios
- postagem realizada por agente;
- uso de `content_source`;
- cadeia de delegações associada;
- arquivo de instruções;
- duas exclusões rápidas.

A convergência dos três incidentes para o SaidIt sustenta a escolha de um único ponto de controle.


In [ ]:
intervention_coverage(df, posts)

## 7. Respostas finais e síntese narrativa

### Q1 — Como o post foi produzido?
Uma cadeia de delegações termina com Chloe Ballast encaminhando a tarefa a John Windward. John publica `SwiftWren.txt` por meio de `content_source`, e os arquivos associados são removidos segundos depois.

### Q2 — Qual é a origem e o significado provável?
A origem operacional é `SwiftWren.txt`; a origem processual envolve `SwiftWren_further_instructions.md` e a propagação entre agentes. O conteúdo literal não é recuperável. A interpretação sustentada é de uma publicação preparada por arquivo, propagada de forma automatizada e removida após a ação.

### Q3a — Houve casos anteriores?
Sim. HiddenOrca e MellowOtter apresentam o mesmo mecanismo final. SwiftWren se destaca pela maior duração e pelo maior número de delegações.

### Q3b — Onde intervir?
No SaidIt, imediatamente antes da publicação, aplicando uma validação baseada na combinação dos sinais observados.

### Padrões, tendência e outlier
- **Padrão:** uso de `content_source`, convergência para John Windward e duas exclusões rápidas;
- **Tendência:** cadeias maiores acumulam mais eventos e maior duração;
- **Outlier:** SwiftWren possui 191 eventos e 186 delegações, superando amplamente os casos anteriores.

### Limitações
Os logs não preservam o texto dos arquivos e não permitem atribuir intenção aos agentes. As conclusões semânticas são, portanto, inferências comportamentais.


## Reprodutibilidade e relação com o dashboard

As funções de carregamento, pré-processamento, análise e construção de visualizações são compartilhadas com o dashboard Streamlit por meio dos módulos em `src/`. Essa separação evita divergências entre o notebook e a aplicação.

O notebook documenta o raciocínio metodológico; o Streamlit oferece dois modos:

- **História guiada:** reproduz a sequência investigativa apresentada neste notebook;
- **Exploração livre:** permite filtrar casos, eventos, agentes e evidências.

As figuras utilizadas na apresentação podem ser regeneradas com:

```bash
python scripts/export_figures.py
```

O dashboard pode ser executado com:

```bash
streamlit run app.py
```
